# CVaR, Backtesting & Market Regime Analysis
**Author:** Sangwon Park (swp0569)

This notebook demonstrates:
1. **CVaR (Expected Shortfall)** vs VaR comparison — why CVaR is a better risk measure for options
2. **Backtesting** — Kupiec POF & Christoffersen independence tests on our MC VaR model
3. **Market Regime Comparison** — how VaR/CVaR behave across stress vs normal periods
4. **Subadditivity Check** — proving VaR breaks subadditivity for options while CVaR doesn't

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.pricing.option_pricer import bs_price
from src.pricing.stock_pricer import calibrate_gbm
from src.risk.monte_carlo_var import monte_carlo_option_var, monte_carlo_stock_var, monte_carlo_portfolio_var
from src.risk.expected_shortfall import cvar_from_pnl, portfolio_cvar
from src.risk.backtesting import kupiec_pof_test, christoffersen_test, backtest_var
from src.risk.parametric_var import parametric_var
from src.volatility.ewma import ewma_volatility
from src.config import DEFAULT_CONFIDENCE, DEFAULT_HORIZON_DAYS, DEFAULT_N_SIM, REGIMES

print('All imports successful.')

In [ ]:
# Load repo data
prices = pd.read_csv('../data/raw/stock_prices.csv', index_col='Date', parse_dates=True)
options = pd.read_csv('../data/raw/option_data.csv')
print(f'Stock prices: {prices.shape[0]} days, tickers: {list(prices.columns)}')
print(f'Options: {len(options)} positions')
print(options)
print(f'\nDate range: {prices.index[0].date()} to {prices.index[-1].date()}')

---
## 1. VaR vs CVaR Comparison
CVaR captures tail risk beyond the VaR threshold. For options with nonlinear payoffs, VaR can underestimate extreme losses.

In [ ]:
# Calibrate from repo data
aapl_cal = calibrate_gbm(prices['AAPL'].dropna().values)
S0_aapl = prices['AAPL'].iloc[-1]

print(f'AAPL Calibration:')
print(f'  mu (ann):    {aapl_cal["mu"]:.4f}')
print(f'  sigma (ann): {aapl_cal["sigma"]:.4f}')
print(f'  skewness:    {aapl_cal["skewness"]:.4f}')
print(f'  kurtosis:    {aapl_cal["kurtosis"]:.4f}')
print(f'  S0:          ${S0_aapl:.2f}')

In [ ]:
# Run MC VaR for AAPL call option (from repo option_data)
opt = options[options['ticker'] == 'AAPL'].iloc[0]
K = opt['strike']
T = opt['maturity']
option_price = opt['market_price']

result = monte_carlo_option_var(
    S0=S0_aapl, K=K, T=T, r=0.05, sigma=aapl_cal['sigma'],
    option_price=option_price, option_type=opt['type'],
    contracts=10, n_sim=50000, confidence=0.99, horizon_days=10
)

# Compute CVaR
es = cvar_from_pnl(result['pnl'], confidence=0.99)

print(f'AAPL {opt["type"].upper()} Option (K={K}, T={T}y, 10 contracts)')
print(f'={"="*50}')
print(f'  99% VaR (10-day):  ${result["var"]:>12,.2f}')
print(f'  99% CVaR (10-day): ${es["cvar"]:>12,.2f}')
print(f'  CVaR/VaR Ratio:    {es["cvar_var_ratio"]:>12.3f}')
print(f'  Tail observations: {es["n_tail"]}')

In [ ]:
# Confidence level comparison
print(f'{"Confidence":<12} {"VaR":>12} {"CVaR":>12} {"CVaR/VaR":>10}')
print('-' * 48)

for conf in [0.90, 0.95, 0.99]:
    r_ = monte_carlo_option_var(
        S0=S0_aapl, K=K, T=T, r=0.05, sigma=aapl_cal['sigma'],
        option_price=option_price, option_type=opt['type'],
        contracts=10, n_sim=50000, confidence=conf, horizon_days=10
    )
    e_ = cvar_from_pnl(r_['pnl'], conf)
    print(f'{conf:<12.0%} ${r_["var"]:>11,.2f} ${e_["cvar"]:>11,.2f} {e_["cvar_var_ratio"]:>10.3f}')

In [ ]:
# Loss distribution with VaR and CVaR
fig, ax = plt.subplots(figsize=(10, 6))
losses = -result['pnl']

ax.hist(losses, bins=100, density=True, alpha=0.7, color='#2C3E50', edgecolor='none')
ax.axvline(es['var'], color='#E74C3C', linestyle='--', lw=2,
           label=f'VaR (99%) = ${es["var"]:,.0f}')
ax.axvline(es['cvar'], color='#E67E22', linestyle='-.', lw=2,
           label=f'CVaR (99%) = ${es["cvar"]:,.0f}')

# Shade tail
tail = losses[losses >= es['var']]
if len(tail) > 0:
    ax.hist(tail, bins=50, density=True, alpha=0.5, color='#E74C3C', edgecolor='none',
            label=f'Tail region ({len(tail)} obs)')

ax.set_xlabel('Loss ($)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('AAPL Call Option — Monte Carlo Loss Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Subadditivity Check
VaR is NOT a coherent risk measure — it can violate subadditivity for nonlinear portfolios.
CVaR always satisfies subadditivity. This is the key reason the TA recommended CVaR.

In [ ]:
# AAPL call + MSFT put portfolio
msft_cal = calibrate_gbm(prices['MSFT'].dropna().values)
S0_msft = prices['MSFT'].iloc[-1]
msft_opt = options[options['ticker'] == 'MSFT'].iloc[0]

# Individual option VaRs
res_aapl = monte_carlo_option_var(
    S0=S0_aapl, K=opt['strike'], T=opt['maturity'], r=0.05,
    sigma=aapl_cal['sigma'], option_price=opt['market_price'],
    option_type=opt['type'], contracts=10, n_sim=50000, confidence=0.99, horizon_days=10
)
res_msft = monte_carlo_option_var(
    S0=S0_msft, K=msft_opt['strike'], T=msft_opt['maturity'], r=0.05,
    sigma=msft_cal['sigma'], option_price=msft_opt['market_price'],
    option_type=msft_opt['type'], contracts=10, n_sim=50000, confidence=0.99, horizon_days=10
)

# Portfolio CVaR with subadditivity check
port_cvar = portfolio_cvar(
    {'AAPL_call': res_aapl['pnl'], 'MSFT_put': res_msft['pnl']},
    confidence=0.99
)

print('Subadditivity Analysis (99% confidence, 10-day horizon)')
print('=' * 60)
print(f'\nVaR:')
print(f'  AAPL call VaR:      ${res_aapl["var"]:>12,.2f}')
print(f'  MSFT put VaR:       ${res_msft["var"]:>12,.2f}')
print(f'  Sum individual:     ${res_aapl["var"] + res_msft["var"]:>12,.2f}')
portfolio_pnl = res_aapl['pnl'] + res_msft['pnl']
portfolio_var = -np.percentile(portfolio_pnl, 1)
print(f'  Portfolio VaR:      ${portfolio_var:>12,.2f}')
print(f'  Subadditive?        {"YES" if portfolio_var <= res_aapl["var"] + res_msft["var"] else "NO — VaR FAILS"}')

print(f'\nCVaR:')
print(f'  AAPL call CVaR:     ${port_cvar["AAPL_call"]["cvar"]:>12,.2f}')
print(f'  MSFT put CVaR:      ${port_cvar["MSFT_put"]["cvar"]:>12,.2f}')
print(f'  Sum individual:     ${port_cvar["sum_individual_cvar"]:>12,.2f}')
print(f'  Portfolio CVaR:     ${port_cvar["portfolio"]["cvar"]:>12,.2f}')
print(f'  Subadditive?        {"YES — CVaR is coherent" if port_cvar["subadditivity_holds"] else "NO"}')

---
## 3. Rolling VaR Backtest
We compute rolling 1-day VaR using historical calibration windows, then compare predicted VaR against actual realized losses.

In [ ]:
# Rolling stock VaR backtest on AAPL using repo data
aapl_prices = prices['AAPL'].dropna()
aapl_returns = np.diff(np.log(aapl_prices.values))

lookback = 252  # 1 year calibration window
n_sim_bt = 10000
confidence = 0.99

bt_dates = []
bt_var = []
bt_cvar = []
bt_actual_loss = []
bt_violations = []

# Start from lookback+1 to have enough history
for i in range(lookback, len(aapl_prices) - 1):
    # Calibrate on trailing window
    window_prices = aapl_prices.values[i - lookback:i]
    cal = calibrate_gbm(window_prices)
    
    S0 = aapl_prices.values[i]
    
    # 1-day stock VaR
    res = monte_carlo_stock_var(
        mu=cal['mu_daily'], sigma=cal['sigma_daily'],
        current_price=S0, shares=100,
        n_sim=n_sim_bt, confidence=confidence, horizon=1, seed=i
    )
    es = cvar_from_pnl(res['pnl'], confidence)
    
    # Actual next-day loss
    actual_pnl = (aapl_prices.values[i + 1] - S0) * 100
    actual_loss = -actual_pnl
    
    bt_dates.append(aapl_prices.index[i])
    bt_var.append(res['var'])
    bt_cvar.append(es['cvar'])
    bt_actual_loss.append(actual_loss)
    bt_violations.append(1 if actual_loss > res['var'] else 0)

bt_df = pd.DataFrame({
    'Date': bt_dates, 'VaR': bt_var, 'CVaR': bt_cvar,
    'Actual_Loss': bt_actual_loss, 'Violation': bt_violations
})

print(f'Backtest period: {bt_df["Date"].iloc[0].date()} to {bt_df["Date"].iloc[-1].date()}')
print(f'Total observations: {len(bt_df)}')
print(f'Violations: {sum(bt_violations)} ({sum(bt_violations)/len(bt_df):.4f})')
print(f'Expected rate: {1 - confidence:.4f}')

In [ ]:
# Statistical tests
kupiec = kupiec_pof_test(sum(bt_violations), len(bt_df), confidence=0.99)
christoffersen = christoffersen_test(bt_violations, confidence=0.99)

print('Backtesting Results')
print('=' * 55)
print(f'\nKupiec POF Test (unconditional coverage):')
print(f'  H0: violation rate = {kupiec["expected_rate"]:.2%}')
print(f'  Observed rate:  {kupiec["observed_rate"]:.4f}')
print(f'  LR statistic:   {kupiec["LR_statistic"]:.4f}')
print(f'  p-value:        {kupiec["p_value"]:.4f}')
print(f'  Result:         {kupiec["conclusion"]}')

print(f'\nChristoffersen Independence Test:')
print(f'  LR statistic:   {christoffersen["LR_statistic"]:.4f}')
print(f'  p-value:        {christoffersen["p_value"]:.4f}')
print(f'  Result:         {christoffersen["conclusion"]}')

In [ ]:
# Backtest visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Top: Actual loss vs VaR/CVaR
axes[0].plot(bt_df['Date'], bt_df['Actual_Loss'], color='#2C3E50', alpha=0.5, lw=0.7, label='Actual Loss')
axes[0].plot(bt_df['Date'], bt_df['VaR'], color='#E74C3C', lw=1.2, label='99% VaR')
axes[0].plot(bt_df['Date'], bt_df['CVaR'], color='#E67E22', lw=1.2, ls='--', label='99% CVaR')

# Mark violations
v_mask = bt_df['Violation'] == 1
axes[0].scatter(bt_df.loc[v_mask, 'Date'], bt_df.loc[v_mask, 'Actual_Loss'],
                color='#E74C3C', s=40, zorder=5, label=f'Violations ({v_mask.sum()})')

axes[0].set_ylabel('Loss ($)', fontsize=11)
axes[0].set_title('AAPL Rolling VaR Backtest (100 shares, 1-day, 99%)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Bottom: Rolling realized vol
aapl_log_ret = np.diff(np.log(aapl_prices.values))
roll_vol = pd.Series(aapl_log_ret).rolling(20).std() * np.sqrt(252)
axes[1].fill_between(aapl_prices.index[1:], roll_vol.values, alpha=0.4, color='#3498DB')
axes[1].set_ylabel('20d Realized Vol', fontsize=11)
axes[1].set_xlabel('Date', fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Market Regime Comparison
Compare VaR and CVaR across different market environments using historical data from the repo.

In [ ]:
# Define regimes using repo's AAPL price data (2020-2025)
# Adapt REGIMES to match our data range
regime_windows = {
    'COVID Crash':       ('2020-02-01', '2020-06-30'),
    'Post-COVID Rally':  ('2020-07-01', '2021-06-30'),
    'Rate Hikes 2022':   ('2022-01-01', '2022-12-31'),
    'Recovery 2023':     ('2023-01-01', '2023-12-31'),
    'Normal 2024':       ('2024-01-01', '2024-12-31'),
}

regime_results = []

for regime_name, (start, end) in regime_windows.items():
    mask = (prices.index >= start) & (prices.index <= end)
    regime_prices = prices.loc[mask, 'AAPL'].dropna()
    
    if len(regime_prices) < 30:
        continue
    
    cal = calibrate_gbm(regime_prices.values)
    S0 = regime_prices.iloc[-1]
    K_atm = round(S0)  # ATM strike
    opt_price = bs_price(S0, K_atm, 30/365, 0.05, cal['sigma'], 'call')
    
    if opt_price < 0.01:
        continue
    
    mc = monte_carlo_option_var(
        S0=S0, K=K_atm, T=30/365, r=0.05, sigma=cal['sigma'],
        option_price=opt_price, option_type='call',
        contracts=10, n_sim=50000, confidence=0.99, horizon_days=10
    )
    es = cvar_from_pnl(mc['pnl'], 0.99)
    
    regime_results.append({
        'Regime': regime_name,
        'S0': round(S0, 2),
        'Realized Vol': round(cal['sigma'], 4),
        'Skewness': round(cal['skewness'], 3),
        'Kurtosis': round(cal['kurtosis'], 3),
        'VaR (99%)': round(mc['var'], 2),
        'CVaR (99%)': round(es['cvar'], 2),
        'CVaR/VaR': round(es['cvar_var_ratio'], 3),
    })

regime_df = pd.DataFrame(regime_results)
regime_df

In [ ]:
# Regime comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(regime_df))
width = 0.35

# VaR vs CVaR
axes[0].bar(x - width/2, regime_df['VaR (99%)'], width, label='VaR', color='#2C3E50')
axes[0].bar(x + width/2, regime_df['CVaR (99%)'], width, label='CVaR', color='#E74C3C')
axes[0].set_xticks(x)
axes[0].set_xticklabels(regime_df['Regime'], rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Loss ($)', fontsize=11)
axes[0].set_title('VaR vs CVaR by Market Regime (99%, 10-day)', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Realized Vol
colors = ['#E74C3C' if v > 0.35 else '#F39C12' if v > 0.25 else '#3498DB' for v in regime_df['Realized Vol']]
axes[1].bar(x, regime_df['Realized Vol'], color=colors, alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(regime_df['Regime'], rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Annualized Volatility', fontsize=11)
axes[1].set_title('Realized Volatility by Regime', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CVaR/VaR ratio across regimes — does tail risk increase during stress?
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(regime_df['Regime'], regime_df['CVaR/VaR'], color='#8E44AD', alpha=0.8)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='CVaR = VaR')
ax.set_ylabel('CVaR / VaR Ratio', fontsize=12)
ax.set_title('Tail Risk Severity by Regime (CVaR/VaR > 1 means fatter tails)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Horizon Comparison (1-day vs 5-day vs 10-day)
Testing how VaR/CVaR scale with horizon — Basel requires 10-day, but Jack suggested 5-day for weekly risk.

In [ ]:
# Horizon comparison
print(f'{"Horizon":<10} {"VaR":>12} {"CVaR":>12} {"CVaR/VaR":>10} {"sqrt-t VaR":>12}')
print('-' * 58)

var_1d = None
for h in [1, 5, 10]:
    r_ = monte_carlo_option_var(
        S0=S0_aapl, K=K, T=T, r=0.05, sigma=aapl_cal['sigma'],
        option_price=option_price, option_type=opt['type'],
        contracts=10, n_sim=50000, confidence=0.99, horizon_days=h
    )
    e_ = cvar_from_pnl(r_['pnl'], 0.99)
    
    if h == 1:
        var_1d = r_['var']
    sqrt_t_var = var_1d * np.sqrt(h) if var_1d else 0
    
    print(f'{h:>3}d{"":<6} ${r_["var"]:>11,.2f} ${e_["cvar"]:>11,.2f} {e_["cvar_var_ratio"]:>10.3f} ${sqrt_t_var:>11,.2f}')

print(f'\nNote: sqrt-t scaling assumes VaR_h = VaR_1 * sqrt(h).')
print(f'Deviation from sqrt-t indicates nonlinear time-scaling of option risk.')

---
## 6. Combined Stock + Option Portfolio
Using the repo's stock + option data to show diversification benefit and asset class decomposition.

In [ ]:
# Portfolio: AAPL stock (100 shares) + AAPL call (10 contracts) + MSFT put (10 contracts)
stock_pos = [
    {'S0': S0_aapl, 'mu': aapl_cal['mu_daily'], 'sigma': aapl_cal['sigma_daily'], 'shares': 100},
]
option_pos = [
    {'S0': S0_aapl, 'K': opt['strike'], 'T': opt['maturity'], 'sigma': aapl_cal['sigma'],
     'option_price': opt['market_price'], 'option_type': opt['type'], 'contracts': 10},
    {'S0': S0_msft, 'K': msft_opt['strike'], 'T': msft_opt['maturity'], 'sigma': msft_cal['sigma'],
     'option_price': msft_opt['market_price'], 'option_type': msft_opt['type'], 'contracts': 10},
]

# AAPL-AAPL corr=1.0, AAPL-MSFT corr from data
aapl_ret = prices['AAPL'].pct_change().dropna()
msft_ret = prices['MSFT'].pct_change().dropna()
corr_am = aapl_ret.corr(msft_ret)
print(f'AAPL-MSFT correlation: {corr_am:.3f}')

corr_matrix = np.array([
    [1.0,     1.0,     corr_am],   # AAPL stock
    [1.0,     1.0,     corr_am],   # AAPL call (same underlying)
    [corr_am, corr_am, 1.0    ],   # MSFT put
])

port = monte_carlo_portfolio_var(
    stock_pos, option_pos, r=0.05, n_sim=50000,
    confidence=0.99, horizon_days=10, corr_matrix=corr_matrix
)
port_es = cvar_from_pnl(port['portfolio_pnl'], 0.99)

print(f'\nPortfolio Risk Decomposition (99%, 10-day)')
print('=' * 50)
print(f'  Portfolio VaR:       ${port["portfolio_var"]:>12,.2f}')
print(f'  Portfolio CVaR:      ${port_es["cvar"]:>12,.2f}')
print(f'  Stock VaR:           ${port["stock_var"]:>12,.2f}')
print(f'  Option VaR:          ${port["option_var"]:>12,.2f}')
print(f'  Sum individual:      ${port["sum_individual_var"]:>12,.2f}')
print(f'  Diversification:     {port["diversification_ratio"]:>12.3f}')
print(f'  Benefit:             ${port["sum_individual_var"] - port["portfolio_var"]:>12,.2f}')

---
## Summary

**Key Findings:**

1. **CVaR > VaR in all scenarios** — CVaR/VaR ratio ranges from ~1.0 to ~1.3+, confirming VaR underestimates tail risk.

2. **CVaR is coherent, VaR is not** — CVaR satisfies subadditivity for our option portfolio, meaning diversification always reduces portfolio CVaR. VaR does not guarantee this.

3. **Regime dependence is significant** — COVID and Rate Hike periods show dramatically higher VaR/CVaR vs normal periods. The CVaR/VaR ratio also increases during stress, meaning tails get fatter.

4. **Backtesting validates the model** — Kupiec and Christoffersen tests provide statistical evidence on whether our MC VaR estimates are well-calibrated.

5. **Horizon scaling is nonlinear for options** — sqrt(t) scaling underestimates multi-day option VaR due to gamma and theta effects.